# 02 — The "aerial river" feeding Santa Cruz

[Open in Colab](https://colab.research.google.com/github/NTU-CompHydroMet-Lab/AguaTrack-ARCO-SA-Tutorial/blob/main/notebooks/02_aerial_river_santa_cruz.ipynb)

**What we're investigating.** Evapotranspiration from the Amazon basin
sustains rainfall in remote downwind regions through "aerial rivers" of
atmospheric moisture. This notebook maps the **30-year climatological
precipitationshed** of Santa Cruz de la Sierra, Bolivia — backward into
the Amazon — to identify which upwind grids supply the city's rainfall,
and how that contribution has shifted across the 1990s, 2000s, and 2010s.

**What you'll get.** Two figures and one inline table:

- `fig1_decadal_precipitationsheds.png`: three side-by-side maps (1990s,
  2000s, 2010s), shared colour scale, with HydroBASINS basin boundaries
  overlaid for visual context.
- `fig2_annual_timeseries.png`: yearly tagged-precipitation bars + a
  5-year moving-average line.
- The top-10 contributing grid cells of the 30-year climatology (rank,
  coordinates, mm/yr, share %), printed inline.

**Dataset.** AguaTrack v1, **yearly aggregate** zarr stores
(`{year}_yearly_sum.zarr`). Same schema as the daily store but with
`time` collapsed to one annual sum. We open one store per year and
slice a single tag cell — minimal memory footprint.

**How to cite.** TODO: paper in submission.

## Step 1 — Configuration

Everything you might want to edit lives in this single cell:

- **HuggingFace dataset** — `AguaTrack/AguaTrack-ARCO-SA`. Yearly
  aggregate sub-directory is not yet uploaded.
- **Target location** — Santa Cruz de la Sierra by default; set
  `TARGET_LAT` / `TARGET_LON` to any other South American city to
  re-run the same analysis there.
- **Year range** — full 1990–2019 record; trim it for a faster Colab
  run if needed.
- **Shapefile source URL** — points at the HydroBASINS bundle that
  ships in this repo. Colab fetches it via the GitHub raw URL.

In [ ]:
HF_REVISION = "main"

TARGET_NAME = "Santa Cruz de la Sierra"
TARGET_LAT = -17.8
TARGET_LON = -63.2
ALL_YEARS = list(range(1990, 2020))

# Full zarr URLs, one per year. The yearly-aggregate sub-directory is
# not yet on HuggingFace — these URLs will start resolving once it is
# uploaded under the same naming scheme.
AGUATRACK_YEARLY_URLS = [
    f"hf://datasets/AguaTrack/AguaTrack-ARCO-SA/AguaTrack_ARCO_SA_yearly/{yr}_yearly_sum.zarr"
    for yr in ALL_YEARS
]

SHP_BASE = "https://github.com/NTU-CompHydroMet-Lab/AguaTrack-ARCO-SA-Tutorial/raw/main/data/hybas_sa_lev02_v1c/"
SHP_DIR = "data/hybas_sa_lev02_v1c"

## Step 2 — Install dependencies and fetch shapefile (Colab only)

Colab gets a fresh runtime every session, so we install the geo stack
and download the HydroBASINS level-2 shapefile (~3.6 MB) from the
GitHub repo. Local users skip both steps — `uv sync` already provided
the deps and the shapefile is on disk under `data/`.

In [ ]:
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from IPython import get_ipython
    get_ipython().run_line_magic(
        "pip",
        'install -q cartopy cmcrameri geopandas "xarray>=2026" "zarr>=3" '
        "fsspec huggingface_hub dask",
    )

    # The .shp alone is not enough — geopandas also needs .shx / .dbf /
    # .prj at minimum (sbn / sbx / xml are optional ESRI indexes).
    import os, urllib.request
    if not os.path.exists(f"{SHP_DIR}/hybas_sa_lev02_v1c.shp"):
        os.makedirs(SHP_DIR, exist_ok=True)
        for ext in ("shp", "shx", "dbf", "prj"):
            urllib.request.urlretrieve(
                f"{SHP_BASE}hybas_sa_lev02_v1c.{ext}",
                f"{SHP_DIR}/hybas_sa_lev02_v1c.{ext}",
            )

## Step 3 — Imports, plotting style, and the basin-overlay shapefile

In [ ]:
from pathlib import Path

import cartopy.crs as ccrs
import cmcrameri.cm as cmc
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import xarray as xr

plt.rcParams.update({
    "font.size": 16,
    "axes.titlesize": 16,
    "axes.labelsize": 16,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 14,
})

basin_gdf = gpd.read_file(f"{SHP_DIR}/hybas_sa_lev02_v1c.shp")

## Step 4 — Find the tag cell nearest to Santa Cruz

We use the 1990 store as a coordinate reference (`tag_lat`/`tag_lon` are
stable across years). A simple Euclidean argmin in (lat, lon) gives us
the nearest tagged grid cell — at 0.25 deg resolution this is precise
enough; no need for a great-circle distance.

In [ ]:

# Open the 1990 store and find the nearest tag in pure xarray.
# `argmin()` returns a 0-D DataArray; cast to int to use as an index.
ds_ref = xr.open_zarr(AGUATRACK_YEARLY_URLS[0],
                      storage_options={"revision": HF_REVISION})
dist_sq = (ds_ref.tag_lat - TARGET_LAT) ** 2 + (ds_ref.tag_lon - TARGET_LON) ** 2
tag_idx = int(dist_sq.argmin())
print(
    f"  target=({TARGET_LAT}, {TARGET_LON})  "
    f"nearest tag=({float(ds_ref.tag_lat.isel(tagging_mask=tag_idx)):.2f}, "
    f"{float(ds_ref.tag_lon.isel(tagging_mask=tag_idx)):.2f})  idx={tag_idx}"
)

## Step 5 — Build a 30-year lazy view, then materialise once

Pangeo idiom: open all 30 yearly stores into one virtual dataset
concatenated along a new `year` dimension, build the *lazy* per-year
source maps and precipitation totals, and trigger a single
`.compute()` at the end with a progress bar. dask handles the
scheduling — no per-year loop in Python.

Memory budget: each yearly source map is a (lat, lon) float32 of
~80k cells. Stacked over 30 years that's ~10 MB — trivially small,
safe even on Colab free tier.

In [ ]:
from dask.diagnostics import ProgressBar  # standard Pangeo progress bar

# Stack 30 lazy datasets along a new "year" dim. The single tag is
# selected before any data is read.
ds_all = xr.concat(
    [
        xr.open_zarr(url, storage_options={"revision": HF_REVISION})
            .isel(tagging_mask=tag_idx)
        for url in AGUATRACK_YEARLY_URLS
    ],
    dim=xr.Variable("year", ALL_YEARS),
)

# `lsm` is shared across years (land-sea mask), so take the first one.
lsm = ds_ref.lsm

# Build the lazy graph: per-year land-only source map + total tagged precip.
year_src_lazy = ds_all.e_track.where(lsm).mean(dim="time")            # (year, lat, lon)
year_precip_lazy = ds_all.tagged_precip.sum(dim="time")               # (year,)

# Materialise both in one go.
with ProgressBar():
    year_src = year_src_lazy.load()
    year_precip = year_precip_lazy.load()

ds_ref.close()
print(f"loaded {year_src.sizes['year']} yearly source maps "
      f"({year_src.nbytes / 1e6:.1f} MB in RAM)")

## Step 6 — Decadal precipitationshed maps

We average the yearly source maps within each decade and plot three
side-by-side maps with a shared colour scale. The red star marks Santa
Cruz; the warm pixels show where its rainfall came from. Notice the
upwind tail extending into the Amazon — that's the "aerial river".

In [ ]:
DECADES = [
    ("1990s", list(range(1990, 2000))),
    ("2000s", list(range(2000, 2010))),
    ("2010s", list(range(2010, 2020))),
]

# Per-decade mean: select the years for each decade and average. Pure
# xarray — no Python loop, no manual concat.
decade_maps = [
    (label, year_src.sel(year=years).mean(dim="year"))
    for label, years in DECADES
]
# 99th percentile across all three decades sets a shared colour-bar max.
# `xr.concat([...]).quantile(0.99)` lets xarray handle NaN propagation.
vmax = float(xr.concat([dm for _, dm in decade_maps], dim="decade").quantile(0.99))
levels = np.linspace(0, vmax, 11)


def style_ax(ax, title, plot_yticks=True):
    """Decorate a single panel: coastlines, extent, gridlines."""
    ax.coastlines(resolution="50m", color="black", linewidth=0.8)
    ax.set_extent([-95, -30, -55, 15], crs=ccrs.PlateCarree())
    gl = ax.gridlines(draw_labels=True, linewidth=0.1)
    gl.top_labels = gl.right_labels = False
    if not plot_yticks:
        gl.left_labels = False
    gl.xlocator = mticker.FixedLocator(np.arange(-180, 181, 10))
    gl.ylocator = mticker.FixedLocator(np.arange(-90, 91, 10))
    ax.set_title(title)


fig, axes = plt.subplots(
    1, 3, figsize=(16, 5.3), subplot_kw={"projection": ccrs.PlateCarree()},
)
cf = None
for col_idx, (ax, (label, dmap)) in enumerate(zip(axes, decade_maps)):
    cf = dmap.plot.contourf(
        ax=ax, levels=levels, cmap=cmc.batlowW,
        transform=ccrs.PlateCarree(), add_colorbar=False,
    )
    basin_gdf.boundary.plot(
        ax=ax, color="dimgray", linewidth=0.5,
        transform=ccrs.PlateCarree(),
    )
    # Red star at Santa Cruz.
    ax.scatter(
        TARGET_LON, TARGET_LAT, color="red", s=120, marker="*",
        transform=ccrs.PlateCarree(), zorder=5, label=TARGET_NAME,
    )
    ax.legend(loc="lower left", fontsize=10)
    style_ax(ax, label, plot_yticks=col_idx == 0)

# One shared horizontal colour bar at the bottom.
fig.subplots_adjust(bottom=0.12, wspace=0.05)
cbar_ax = fig.add_axes([0.15, 0.02, 0.7, 0.025])
fig.colorbar(cf, cax=cbar_ax, orientation="horizontal",
             label="Mean Moisture Contribution (mm/yr)")
fig.suptitle(f"Decadal Precipitationsheds — {TARGET_NAME}", fontsize=14)

OUT = Path("outputs/aerial_river"); OUT.mkdir(parents=True, exist_ok=True)
fig.savefig(OUT / "fig1_decadal_precipitationsheds.png", bbox_inches="tight", dpi=150)
plt.show()

## Step 7 — Annual time series with a 5-year moving average

Bars show how much tagged rain reached Santa Cruz each year. The dark
red line is a 5-year moving average — smooths out year-to-year noise so
the long-run trend is easier to read. We trim the first and last two
years from the moving-average plot because those points are biased by
the boundary of the convolution window.

In [ ]:
# 5-yr centred rolling mean — pure xarray, NaN at the boundaries by default.
ma = year_precip.rolling(year=5, center=True).mean()

fig, ax = plt.subplots(figsize=(16, 7.3))
ax.bar(year_precip.year.values, year_precip.values,
       color="steelblue", alpha=0.7, label="Annual")
ax.plot(ma.year.values, ma.values, color="darkred", lw=2, label="5-yr moving avg")
ax.set_xlabel("Year")
ax.set_ylabel("Tagged Precipitation (mm/yr)")
ax.set_title(f"Annual Moisture Received — {TARGET_NAME}")
ax.legend()
ax.set_xlim(1989, 2020)
fig.savefig(OUT / "fig2_annual_timeseries.png", bbox_inches="tight", dpi=150)
plt.show()

## Step 8 — Top-10 contributing grid cells and decadal trend

We rank the pixels of the 30-year climatology by their contribution and
print the top 10. We also report decadal means and a simple linear
regression slope (1990–2019).

In [ ]:
# 30-year climatology = mean over the year axis. Pure xarray.
clim_map = year_src.mean(dim="year")

# Top-10 contributing pixels via xarray stacking. We stack lat+lon into
# one flat dim, drop NaN/zero pixels, sort descending, and take the
# top 10. This avoids the np.repeat / np.tile gymnastics of flattening
# a numpy array by hand.
clim_flat = (
    clim_map.stack(cell=("latitude", "longitude"))
    .where(lambda da: da > 0, drop=True)
)
top10 = clim_flat.sortby(clim_flat, ascending=False).isel(cell=slice(10))
total_clim = float(clim_map.sum())

print(f"\nTop 10 contributing grid cells (30-yr climatology, total = {total_clim:.1f} mm/yr):\n")
print(f"{'Rank':<6}{'Latitude':<12}{'Longitude':<12}{'Contribution':<18}{'Share':<8}")
for i in range(top10.sizes["cell"]):
    lat = float(top10.latitude[i])
    lon = float(top10.longitude[i])
    v = float(top10[i])
    print(f"{i+1:<6}{lat:+.2f}{'':<6}{lon:+.2f}{'':<6}{v:.3f} mm/yr{'':<6}{v/total_clim*100:.2f}%")

# Decadal means via xarray slicing on the `year` coordinate.
d1 = float(year_precip.sel(year=slice(1990, 1999)).mean())
d2 = float(year_precip.sel(year=slice(2000, 2009)).mean())
d3 = float(year_precip.sel(year=slice(2010, 2019)).mean())

# Linear trend via xarray polyfit. Returns coefficients on a `degree`
# axis (deg=1 -> [intercept, slope] for degree=[0, 1]).
fit = year_precip.polyfit(dim="year", deg=1)
slope = float(fit.polyfit_coefficients.sel(degree=1))

print(f"\nDecadal means (mm/yr):  1990s={d1:.1f}  2000s={d2:.1f}  2010s={d3:.1f}")
print(f"Linear trend 1990–2019: {slope:+.3f} mm/yr per year ({slope * 10:+.2f} mm/decade)")